<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Solution: Decide, review or send to a person

**[Decide, review or send to a person](https://learning.nextia-ai.com/courses/text-routing/m08/decide-review-or-send-to-a-person/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 8, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** split the messages into three zones by confidence (decide, review, send to a person), choose the thresholds on the validation messages under the review load that the team can carry, and check them once on the test messages.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The cost per 1,000 messages, from [The cost of each mistake](https://learning.nextia-ai.com/courses/text-routing/m01/the-cost-of-each-mistake/). Calibration, from [Is the confidence true?](https://learning.nextia-ai.com/courses/text-routing/m08/is-the-confidence-true/). |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m08/decide-review-or-send-to-a-person/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M08-L02-decide-review-or-send-to-a-person/decide-review-or-send-to-a-person-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield inbox of this course (about 9 MB) into a folder `larkfield-inbox/data`, and the course's helper `routing.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 10000 messages (6000 train, 2000 validation, 2000 test), all synthetic.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-inbox` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield inbox is synthetic: we made it for this course. It has no real people. The answers of the hosted models were recorded once, so you need no key.

In [ ]:
# Setup: download the course data into larkfield-inbox/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/"
FILES = {
    "data/corrections.csv": "be3088aecd9b7ba5d07ea222b79e06a476806eba81bc80c2f33da695878e5cc0",
    "data/costs.json": "9a2d566f062c3382b8884ef07756f886179c69a3525d80e3d4c15c0c4fd4507e",
    "data/dataset-card.md": "ec1d7769d1d68a9e51a5b12660364c74352af936df79533a0d2170867171bdd1",
    "data/double_labelled.csv": "ba4a8dadc3c1ba8d48af893124bd33fb7a895f917a2add211486e3dbd353dfc9",
    "data/guidelines.md": "04df0cd72da4e6b7e6aa629ca21205e5395df7b66aae9268618ae14597e45782",
    "data/launch_labels_sample.csv": "600c689325b7c14e56f4a27e92e395eb0b2e965411b1abf9d724720c8b485029",
    "data/launch_month.csv": "7ef3cc2c5fa910023c6330c1e895744358846ce451e551b86be8c9abe1c4289f",
    "data/messages.csv": "266ca23c0551249fc114fc74253103ea8602a04be1fe31d1b7f91bc94ce6dd5e",
    "data/old_queue.csv": "e3cde5d19d3ebe2e6e0e067693c76796c0abbf89df0dc93dc310ad3fe42a5cae",
    "data/predictions/embeddings-newer.csv": "3c7ffe5b5371ca60c3b6bedc17b307142bcfd77f06c97db7ed267b703a50a8d2",
    "data/predictions/embeddings-newer.json": "dcd537f2971b9a48b5354cc9fbb48b87689b83b3a479da9acd119701c2fc19f6",
    "data/predictions/embeddings-older.csv": "a06c490ac17cbc1c3c75380f6cdf7775483de92961cdbb78b2a7914256926539",
    "data/predictions/embeddings-older.json": "db658bbd13861529122807726b6b41e1da9c70e3b55aba55cce9e868a795e8b4",
    "data/predictions/encoder-finetuned.csv": "abced2276671bfd413319573907a48c60952310e50978dba96c0dde0332b7d39",
    "data/predictions/encoder-finetuned.json": "1f7c89271a199c1d659bdc29d8ac0592558a1424218bd0519bbbde9840212b94",
    "data/predictions/gliclass-zero-shot.csv": "2c22e5036de94a376f7c0ccf22603c4d7c624135c8a4965123c9492748814008",
    "data/predictions/gliclass-zero-shot.json": "95ad2acafd48ad2c24516333dba7e78a3eabc7ca616c4053ce2f144f56cbaaec",
    "data/predictions/llm-few-shot.csv": "bcdaee3550f78ebb672465314bbcc9ad26283aaea69265a979739cf6a5f965e2",
    "data/predictions/llm-few-shot.json": "104b446894a0450a7e02fcd07c6db44f3e1a5486002b388374ce6dae4a744a12",
    "data/predictions/llm-zero-shot.csv": "5518e2b668a110b572687313bc13bd689774dff96c0a1614e1cd92e14ddf9ce7",
    "data/predictions/llm-zero-shot.json": "9a72ce600d0d858b00dd3b37247a305f9e8b3fc2fef3b8dbe60c0871b0c1b355",
    "data/predictions/local-llm-prompted.csv": "5bb0d84f8ee9b5086818d3525e1da3f8172b38b7875b10d1d3021b3ad5889f33",
    "data/predictions/local-llm-prompted.json": "ae328379d5ea294b3b59845ee143f24776787e748bc14c165b5e8e9ed4ff7158",
    "data/predictions/local-llm-scores.csv": "f85f570075ea9288fd14a6c3955d4310ba740814f3de472e8a263cd4f9682a5f",
    "data/predictions/local-llm-scores.json": "67a7e07ce065934a947f59c1c36d416eec76f8827f01500ce82ba6e6c4d9b99e",
    "data/predictions/nli-zero-shot.csv": "ecafb001370e5cf2904ce685683beafc86a2c290cd1da80438abb9912d099d3c",
    "data/predictions/nli-zero-shot.json": "54c2e724cb3c855a0fa6302d4ac0ab2df2ffaf788eec4ab198c6feb8d3508928",
    "data/prices.json": "9e98f9056d9d1e53d76a039da9b10d2b4fa54c8a5b35eb6cae32a2abdee7af38",
    "data/recordings/README.md": "d99eb50666045cc2ccf596e100103244fe1513b488fce46e26462a2666053ffc",
    "data/recordings/llm-few-shot.jsonl": "259a93f1d6dcbcb167cd45796c8e09a914e56c7741f14bd0aee53c996e8349d4",
    "data/recordings/llm-labels.jsonl": "cd6c670a23ee78cd3d8b51e5907ebae9d86448aa6236767c11d55227616fa8cc",
    "data/recordings/llm-reasoning.jsonl": "fc8ae631121f3240ce378df429af767ac52ea6e1560aa409de09ffc99b092300",
    "data/recordings/llm-zero-shot.jsonl": "957edfb8811e177c83bf23772f0da561659cf50728ffc95597f85cd7680aed86",
    "data/rules.json": "6f95d5e7b87869ec7975b198a3777cdd96a0db6a869554d922889479c87b2066",
    "routing.py": "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-inbox").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-inbox and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
import csv
with open("data/messages.csv", encoding="utf-8") as f:
    splits = [row["split"] for row in csv.DictReader(f)]
print(f"Ready: {len(splits)} messages ({splits.count('train')} train, {splits.count('validation')} validation, "
      f"{splits.count('test')} test), all synthetic.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course helper as `rt`, reads the three parts of the inbox (`train`, `validation`, `test`), and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

messages = pd.read_csv("data/messages.csv")
train = messages[messages["split"] == "train"].reset_index(drop=True)
validation = messages[messages["split"] == "validation"].reset_index(drop=True)
test = messages[messages["split"] == "test"].reset_index(drop=True)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The model and its confidence

The next cell trains TF-IDF + logistic regression on the 6,000 training messages, as the project does, and keeps the probabilities of every route for the validation and test messages. The **confidence** of a message is its top probability. It takes about 5 seconds.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, make_pipeline

lr = make_pipeline(
    FeatureUnion([("words", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")),
                  ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True))]),
    LogisticRegression(C=10, max_iter=3000)).fit(train["text"], train["route"])
p_val = pd.DataFrame(lr.predict_proba(validation["text"]), columns=lr.classes_)
p_test = pd.DataFrame(lr.predict_proba(test["text"]), columns=lr.classes_)
print("validation confidence: median", round(p_val.max(axis=1).median(), 3))

## 2. Three zones

Two thresholds make three zones:

- **decide**: the confidence is `high` or more. The model's route is used.
- **review**: the confidence is between `low` and `high`. An agent reads the message and sends it on (US$1.50).
- **person**: the confidence is under `low`. A senior person reads it first, as for a `needs_person` message.

A message that the model routes to `needs_person` also goes to a person: that route **is** "a person reads it first". So the person zone holds both.

`report()` gives the share of each zone, the share of wrong routes in the decide zone, and the cost per 1,000 messages with the course's cost table.

In [ ]:
def zones(p, high, low):
    """The route if the confidence is high or more; "review" between low and high; "needs_person" under low."""
    top, route = p.max(axis=1), p.idxmax(axis=1)
    return np.where(top >= high, route, np.where(top >= low, "review", "needs_person"))

def report(truth, z):
    z = pd.Series(z, index=truth.index)
    auto = ~z.isin(["review", "needs_person"])
    return {"decide": round(auto.mean(), 3), "review": round((z == "review").mean(), 3),
            "person": round((z == "needs_person").mean(), 3),
            "errors_in_decide": round((z[auto] != truth["route"][auto]).mean(), 4),
            "cost_per_1000": round(rt.cost_per_1000(truth, z), 1)}

report(validation, zones(p_val, high=0.80, low=0.40))

## 3. Choose the thresholds on validation, under a review limit

Grace's team can read at most 10% of the messages: the review zone and the person zone together. The next cell tries every `high` from 0.30 to 1.00 in steps of 0.01, with `low` at 0 or at a few values under `high`. It keeps the cheapest pair that respects the limit, **on the validation messages**. Then it reports that pair once on the test messages. It takes a few seconds.

In [ ]:
GRID = np.round(np.arange(0.30, 1.0001, 0.01), 2)

def choose(truth, p, review_max):
    best = None
    for high in GRID:
        for low in np.concatenate([[0.0], GRID[GRID < high][::5]]):
            r = report(truth, zones(p, high, low))
            if r["review"] + r["person"] > review_max:
                continue
            if best is None or r["cost_per_1000"] < best[2]["cost_per_1000"]:
                best = (float(high), float(low), r)
    return best

high, low, on_val = choose(validation, p_val, review_max=0.10)
on_test = report(test, zones(p_test, high, low))
print(f"decide at {high:.2f} or more, person below {low:.2f}")
print("validation:", on_val)
print("test:      ", on_test)

> **Check.** You should see `decide at 0.58 or more, person below 0.00`, a cost of 305.8 on validation and 148.2 on test.

The same thresholds cost twice as much on validation as on test. Both have 2,000 messages. The next cell counts the messages of each part by their cost in dollars: 1.5 is a review, 2 a team's message sent to a person, 4 a wrong queue, 20 an urgent one, 40 a `needs_person` message sent to a team, 200 a missed account takeover.

In [ ]:
for name, truth, p in [("validation", validation, p_val), ("test", test, p_test)]:
    z = zones(p, high, low)
    cost = pd.Series([rt.message_cost(t, q, pr, tp) for t, q, pr, tp in
                      zip(truth["route"], z, truth["priority"], truth["topic"])])
    print(f"{name:10}:", {float(k): int(v) for k, v in cost[cost > 0].value_counts().sort_index().items()})

> **Check.** On validation you should see one message at 200.0: one missed account takeover is US$100 per 1,000 messages on its own.

## 4. The work in each zone

The history has about 65 messages a day. The next cell draws the confidence of the test messages in 20 bins, coloured by zone, and turns the zone shares into messages a day.

In [ ]:
import matplotlib.pyplot as plt

z = zones(p_test, high, low)
top = p_test.max(axis=1).to_numpy()
kind = np.where(z == "review", "review", np.where(z == "needs_person", "person", "decide"))
bins = np.linspace(0, 1, 21)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist([top[kind == k] for k in ["decide", "person", "review"]], bins=bins, stacked=True,
        label=["decide", "person (route needs_person)", "review"], color=["#0054D8", "#4A5A76", "#C2410C"])
ax.axvline(high, ls="--", color="black")
ax.set_ylim(0, 250)
ax.set_xlabel("confidence (top probability)")
ax.set_ylabel("messages (axis cut at 250)")
ax.legend()
plt.show()
per_day = 10000 / 153
for k in ["decide", "review", "person"]:
    print(f"{k:7} {(kind == k).mean():6.1%}  about {(kind == k).mean() * per_day:4.1f} messages a day")

> **Check.** You should see decide 89.7%, review 5.5% and person 4.8%: about 3.6 and 3.1 messages a day for review and person, about 7 together. The project prints 5.6% for review, because it rounds 0.0555 up.

## 5. Your turn: a team that can read 20%

> **Your turn.** Grace hires a second agent, and the team can now read 20% of the messages. Choose the thresholds again on the validation messages with `review_max=0.20`. Put the decide threshold in `high_20`.

In [ ]:
high_20, low_20, on_val_20 = choose(validation, p_val, review_max=0.20)
print(high_20, low_20, on_val_20)

In [ ]:
expect('high_20', high_20, 0.74, tolerance=1e-09)

In [ ]:
if high_20 is ...:
    print("Write your answer in the cell above first.")
else:
    print("validation:", report(validation, zones(p_val, high_20, 0.0)))
    print("test:      ", report(test, zones(p_test, high_20, 0.0)))

> **Check.** With `high_20` = 0.74, the cost is 220.0 on validation and 197.5 on test.

On validation, the larger limit is cheaper (220.0 against 305.8). On test, it is more expensive (197.5 against 148.2). The two parts disagree because a few expensive mistakes decide the cost. Choose on validation, report test once, and say how noisy the number is.

## 6. Failure case: choosing on the test messages

Tomás chooses the thresholds on the test messages, because "that is where we measure".

In [ ]:
h_t, l_t, on_test_t = choose(test, p_test, review_max=0.10)
print(f"chosen on test: decide at {h_t:.2f}; test {on_test_t['cost_per_1000']}; "
      f"validation {report(validation, zones(p_val, h_t, l_t))['cost_per_1000']}")

> **Check.** You should see `decide at 0.57`, a test cost of 144.2 and a validation cost of 311.8.

144.2 looks better than 148.2. But it is the best of about 600 tries on the same 2,000 messages, so it is too good to be true for the next 2,000. The validation messages, which the choice did not see, say 311.8. The test messages are for one honest report at the end, not for choosing.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m08/decide-review-or-send-to-a-person/) has the diagrams, the explorer and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Learn from corrections](https://learning.nextia-ai.com/courses/text-routing/m08/learn-from-corrections/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m08/decide-review-or-send-to-a-person/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.